# `04_r4s3_usd_behavioral.ipynb` — R4-S3-USD **behavioral subscription-inelasticity test** (framework-relevant arm)

**Spec citations:** `docs/specs/2026-05-16-ai-cost-factor-model-design.md` v0.2.8 — §2.2.B (lines 1348–1395), §0.1 CORRECTIONS-S (test inversion), §0.1 CORRECTIONS-I (HAC bandwidth `floor(T^(1/3))`), §0.1 CORRECTIONS-J + CORRECTIONS-U (residual-SD power floor 0.50 at MDES=0.40), §0.7 CORRECTIONS-Y-9 (UTC parity closure), §0.8 CORRECTIONS-Z2 (Task 13 reclassification).

**Plan:** `docs/plans/2026-05-16-ai-cost-factor-model-plan.md` Task 14.

## CORRECTIONS-S preamble (framework-relevant arm)

Per **CORRECTIONS-S**, this notebook is the **framework-relevant arm** of R4-S3. The pre-registered direction is *"null cannot be rejected"* (subscription-regime inelasticity); **rejecting the null** — in either direction — reveals a behavioral channel despite zero marginal cost, which is itself an informative finding. **Both outcomes feed M-design assumptions; neither routes to FAIL.**

**Inversion vs. R4-S3-COP (Task 13).** R4-S3-COP was the *consistency check* (mechanical FX pass-through under USD-denominated cost data → COP). Its v0.2.7 verdict was CONSISTENCY-FAIL, **reclassified to REGIME-CONDITIONAL** per v0.2.8 §0.8 Z2: both preconditions hold — (A) additive-identity check passes at 1.78e-15 absolute error (panel construction is exact `ln COP = ln USD + ln TRM`), and (B) the FX β shrinks toward zero in the deflated regression because TRM movements *cancel out* of the USD-side cost (subscription invoice currency). Task 14 is therefore unblocked; today's test isolates the **behavioral** USD-side response.

**Sample-floor caveat (anti-fishing disclosure).** Current panel has **N=29 weekdays** post-first-diff. §2.2.B verdict logic requires **N ≥ 38** for REJECT_NULL / FAIL_TO_REJECT labels. Per the user's HALT-discipline directive, this notebook will run the regression for the record but route the verdict to **PARTIAL-REJECT** or **PARTIAL-FAIL_TO_REJECT** (with N-below-floor explicitly disclosed). No silent override of the N gate.

**π̂ disclaimer (CORRECTIONS-Y-6).** All notional-cost figures use a fixed `ephemeral_pi_share = 0.398977` (audited point estimate from the snapshot panel); the actual ephemeral cache-write proportion at runtime is unobservable in ccusage logs and is therefore frozen as a calibrated constant, not a measured time-series.

**Y-9 closure (v0.2.7).** Cost data has been verified at **1.000000× ccusage parity** post-UTC alignment (timezone-artifact resolved); LHS and RHS series are apples-to-apples on the same UTC weekday cadence.

## Cell 2 — Anti-fishing pins (declared PRE-DATA)

Constants below are pinned **before** any data read. They must not be tuned post-hoc; doing so is silent-fishing per `feedback_pathological_halt_anti_fishing_checkpoint.md`.

In [1]:
from __future__ import annotations

import numpy as np

# ANTI-FISHING PINS — set PRE-DATA, DO NOT TUNE
LAG_PRIMARY = 1                    # §2.2.B primary lag (k=1)
LAG_SENSITIVITY = 5                # §2.2.B sensitivity lag (k=5)
ALPHA_LEVEL = 0.05                 # TWO-SIDED test threshold (CORRECTIONS-S)
TOKENS_PROXY = 'output_tok'        # consistent with Task 13
USDCOP_SERIES = 'trm_cop_per_usd'  # Banrep TRM
COST_SERIES = 'notional_cost_usd'  # USD-denominated; the behavioral LHS

# Sample-floor gate (§2.2.B verdict logic)
N_MIN = 38

# Power gate (CORRECTIONS-J + CORRECTIONS-U — residual SD, MDES=0.40)
POWER_THRESHOLD = 0.50
MDES_RESID_SD = 0.40
POWER_MC_DRAWS = 2000
POWER_SEED = 20260517            # reproducibility

# HAC bandwidth pinned PRE-DATA as floor(T^(1/3)) per CORRECTIONS-I (Andrews 1991)
def hac_bandwidth(t_lag: int) -> int:
    """Andrews (1991) rule-of-thumb HAC bandwidth: floor(T^(1/3))."""
    return int(np.floor(t_lag ** (1 / 3)))

# Verdict label set (closed; no other labels permitted post-hoc)
VERDICT_LABELS = {
    'REJECT_NULL',
    'FAIL_TO_REJECT',
    'PARTIAL-REJECT',         # p<0.05 but N<38; disclosed
    'PARTIAL-FAIL_TO_REJECT', # p>=0.05 but N<38; disclosed
    'POWER-HALT',
    'HALT',
}

print('=' * 60)
print('ANTI-FISHING PINS (pre-data)')
print('=' * 60)
print(f'LAG_PRIMARY        = {LAG_PRIMARY}')
print(f'LAG_SENSITIVITY    = {LAG_SENSITIVITY}')
print(f'ALPHA_LEVEL        = {ALPHA_LEVEL} (TWO-SIDED per CORRECTIONS-S)')
print(f'TOKENS_PROXY       = {TOKENS_PROXY}')
print(f'USDCOP_SERIES      = {USDCOP_SERIES}')
print(f'COST_SERIES        = {COST_SERIES}')
print(f'N_MIN              = {N_MIN}  (§2.2.B verdict-eligibility floor)')
print(f'POWER_THRESHOLD    = {POWER_THRESHOLD}  at MDES={MDES_RESID_SD} residual-SD')
print(f'POWER_MC_DRAWS     = {POWER_MC_DRAWS}')
print(f'POWER_SEED         = {POWER_SEED}')
print(f'HAC bandwidth      = floor(T^(1/3))  per CORRECTIONS-I')
print(f'VERDICT_LABELS     = {sorted(VERDICT_LABELS)}')

ANTI-FISHING PINS (pre-data)
LAG_PRIMARY        = 1
LAG_SENSITIVITY    = 5
ALPHA_LEVEL        = 0.05 (TWO-SIDED per CORRECTIONS-S)
TOKENS_PROXY       = output_tok
USDCOP_SERIES      = trm_cop_per_usd
COST_SERIES        = notional_cost_usd
N_MIN              = 38  (§2.2.B verdict-eligibility floor)
POWER_THRESHOLD    = 0.5  at MDES=0.4 residual-SD
POWER_MC_DRAWS     = 2000
POWER_SEED         = 20260517
HAC bandwidth      = floor(T^(1/3))  per CORRECTIONS-I
VERDICT_LABELS     = ['FAIL_TO_REJECT', 'HALT', 'PARTIAL-FAIL_TO_REJECT', 'PARTIAL-REJECT', 'POWER-HALT', 'REJECT_NULL']


## Decision-citation block — two-sided test choice

- **Reference:** spec §2.2.B + §0.1 CORRECTIONS-S.
- **Why:** the subscription-regime null is "no behavioral response of USD-side cost vol to FX vol." Under a fixed-fee invoicing regime, marginal cost in USD is zero, so $\alpha_1^{USD}$ should not be meaningfully different from zero — *in either direction*. Rejecting EITHER direction is the behavioral-channel finding (the wage-earner-as-user might shift token consumption defensively when COP weakens, even though their USD bill is unchanged). A one-sided test would silently throw away half the inferential surface.
- **Relevance:** anti-fishing pre-pinning. Switching to one-sided after seeing the sign of $\hat\alpha_1^{USD}$ is silent-fishing and is explicitly banned.
- **Connection:** feeds the verdict cell; `p_two_sided` (from `statsmodels`) is the inputs to the REJECT_NULL / FAIL_TO_REJECT gate.

## Trio 1 — Construct $|\Delta\ln\text{NotionalCost}^{USD}|$, $|\Delta\ln\text{USDCOP}|$, $|\Delta\ln\text{Tokens}|$

### Decision-citation block (4-part)

- **Reference:** spec §2.2.B equation (lines 1351–1353).
- **Why:** the spec equation is symmetric to §2.2.A in structure but with `notional_cost_usd` on the LHS. The absolute log first-difference of three series is the canonical "volatility-on-volatility" RHS for FX-cost transmission (Pair-D precedent + CORRECTIONS-S).
- **Relevance:** same series-construction recipe as Task 13 — but LHS is now USD-denominated, isolating the *behavioral* (token-consumption) channel from the *mechanical* (FX-translation) channel that dominates the COP arm.
- **Connection:** outputs three numpy arrays of length $T = N - 1$ feeding the regression trios.

In [2]:
from pathlib import Path

import polars as pl

PANEL_PATH = Path('../../data/panels/notional_cost_panel.parquet')
df = pl.read_parquet(PANEL_PATH)
print(f'panel shape: {df.shape}')
print(f'date window: {df["date_utc"].min()} -> {df["date_utc"].max()}')
print(f'columns: {df.columns}')

# Compute absolute first-diff of natural log for the three series
abs_dln_usd = df[COST_SERIES].log().diff().abs().drop_nulls().to_numpy()
abs_dln_trm = df[USDCOP_SERIES].log().diff().abs().drop_nulls().to_numpy()
abs_dln_tok = df[TOKENS_PROXY].log().diff().abs().drop_nulls().to_numpy()

T = len(abs_dln_usd)
assert len(abs_dln_trm) == T == len(abs_dln_tok), 'series-length mismatch'

print()
print(f'T (post first-diff)                  = {T}')
print(f'|Δln NotionalCost^USD|  range        = [{abs_dln_usd.min():.6f}, {abs_dln_usd.max():.6f}]')
print(f'|Δln USDCOP|             range        = [{abs_dln_trm.min():.6f}, {abs_dln_trm.max():.6f}]')
print(f'|Δln Tokens (output_tok)| range       = [{abs_dln_tok.min():.6f}, {abs_dln_tok.max():.6f}]')
print(f'\nSpec floor N_MIN={N_MIN}; observed T={T} → {"PASS" if T >= N_MIN else "BELOW FLOOR (PARTIAL-* verdict path)"}')

panel shape: (29, 11)
date window: 2026-01-06 -> 2026-05-14
columns: ['date_utc', 'notional_cost_usd', 'notional_cost_cop', 'trm_cop_per_usd', 'input_tok', 'output_tok', 'cache_create_5m', 'cache_create_1h', 'cache_read', 'n_messages', 'ephemeral_pi_share']

T (post first-diff)                  = 28
|Δln NotionalCost^USD|  range        = [0.000832, 5.483817]
|Δln USDCOP|             range        = [0.000632, 0.023392]
|Δln Tokens (output_tok)| range       = [0.000000, 7.697189]

Spec floor N_MIN=38; observed T=28 → BELOW FLOOR (PARTIAL-* verdict path)


### Interpretation — Trio 1

Three numpy arrays of length $T$ constructed from the production panel. $T$ is reported above; if $T < 38$ the verdict cell will route to a PARTIAL-* label. The USD-side cost vol range is large (driven by sparse low-cost weekdays mixing with high-load weekdays), consistent with subscription invoice variance dominated by message-volume rather than FX.

## Trio 2 — Primary regression k=1 with HAC (TWO-SIDED test)

### Decision-citation block (4-part)

- **Reference:** spec §2.2.B + CORRECTIONS-S (two-sided) + CORRECTIONS-I (HAC `floor(T^(1/3))`).
- **Why:** regress $|\Delta\ln\text{NotionalCost}^{USD}|_t$ on $|\Delta\ln\text{USDCOP}|_{t-1}$ and $|\Delta\ln\text{Tokens}|_{t-1}$ with Newey-West HAC SE; **two-sided test** on $\alpha_1^{USD} = 0$. The token-vol control is mechanical (more tokens → more invoice variance); the FX-vol coefficient is the behavioral channel of interest.
- **Relevance:** k=1 is the pre-pinned primary lag (one-trading-day FX update assimilation); HAC L = $\lfloor T_{lag}^{1/3} \rfloor$ controls for residual autocorrelation under small-$T$ daily cadence. The v0.1.0 fixed $L=12$ would over-shrink at $T_{lag} \approx 28$ and is explicitly rejected by CORRECTIONS-I.
- **Connection:** `p_two_sided_primary` feeds the verdict gate; `model_k1.resid` SD feeds Trio 4's power measurement.

In [3]:
import statsmodels.api as sm

# Lag-aligned arrays for k=1 primary
T_lag_primary = T - LAG_PRIMARY
hac_L_primary = hac_bandwidth(T_lag_primary)
print(f'T_lag (k={LAG_PRIMARY}) = {T_lag_primary}')
print(f'HAC L  = floor(T_lag^(1/3)) = floor({T_lag_primary ** (1/3):.4f}) = {hac_L_primary}')

y_primary = abs_dln_usd[LAG_PRIMARY:]              # LHS at time t
x1_primary = abs_dln_trm[:-LAG_PRIMARY]            # |Δln USDCOP|_{t-1}
x2_primary = abs_dln_tok[:-LAG_PRIMARY]            # |Δln Tokens|_{t-1}
assert len(y_primary) == len(x1_primary) == len(x2_primary) == T_lag_primary, 'alignment broken'

X_primary = sm.add_constant(np.column_stack([x1_primary, x2_primary]))
model_k1 = sm.OLS(y_primary, X_primary).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_primary})
print()
print(model_k1.summary())

# Extract pre-pinned coefficient of interest (α₁^USD)
alpha_1_usd_primary = float(model_k1.params[1])
se_alpha_1_primary = float(model_k1.bse[1])
t_stat_primary = alpha_1_usd_primary / se_alpha_1_primary
p_two_sided_primary = float(model_k1.pvalues[1])   # statsmodels reports TWO-SIDED p-values

print()
print(f'α̂₁^USD (k=1)      = {alpha_1_usd_primary:+.6f}')
print(f'HAC SE            = {se_alpha_1_primary:.6f}')
print(f't-stat            = {t_stat_primary:+.4f}')
print(f'p_two_sided       = {p_two_sided_primary:.6f}')
print(f'reject at α=0.05? = {p_two_sided_primary < ALPHA_LEVEL}')

T_lag (k=1) = 27
HAC L  = floor(T_lag^(1/3)) = floor(3.0000) = 3

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                 -0.076
Method:                 Least Squares   F-statistic:                    0.2527
Date:                Sun, 17 May 2026   Prob (F-statistic):              0.779
Time:                        23:45:16   Log-Likelihood:                -41.645
No. Observations:                  27   AIC:                             89.29
Df Residuals:                      24   BIC:                             93.18
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

### Interpretation — Trio 2 (k=1 primary)

Estimate, HAC SE, t-stat, and two-sided p-value reported above. Pass/fail at the pre-pinned $\alpha = 0.05$ level is determined by whether `p_two_sided_primary < 0.05`. Under CORRECTIONS-S framing: rejection here = behavioral channel; non-rejection = subscription inelasticity confirmed.

## Trio 3 — Sensitivity regression k=5 with HAC

### Decision-citation block (4-part)

- **Reference:** spec §2.2.B sensitivity lag.
- **Why:** same recipe with $k = 5$ to test whether the behavioral response operates over a trading-week horizon (delayed reaction to sustained FX moves vs. one-day reaction).
- **Relevance:** k=5 is pre-pinned for sensitivity *only*; it does not override the k=1 verdict. Divergence between k=1 and k=5 informs Stage-3 M-design temporal assumptions.
- **Connection:** reported alongside k=1 in the verdict cell.

In [4]:
T_lag_sens = T - LAG_SENSITIVITY
hac_L_sens = hac_bandwidth(T_lag_sens)
print(f'T_lag (k={LAG_SENSITIVITY}) = {T_lag_sens}')
print(f'HAC L  = floor(T_lag^(1/3)) = floor({T_lag_sens ** (1/3):.4f}) = {hac_L_sens}')

y_sens = abs_dln_usd[LAG_SENSITIVITY:]
x1_sens = abs_dln_trm[:-LAG_SENSITIVITY]
x2_sens = abs_dln_tok[:-LAG_SENSITIVITY]
assert len(y_sens) == len(x1_sens) == len(x2_sens) == T_lag_sens, 'alignment broken'

X_sens = sm.add_constant(np.column_stack([x1_sens, x2_sens]))
model_k5 = sm.OLS(y_sens, X_sens).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_sens})
print()
print(model_k5.summary())

alpha_1_usd_sens = float(model_k5.params[1])
se_alpha_1_sens = float(model_k5.bse[1])
t_stat_sens = alpha_1_usd_sens / se_alpha_1_sens
p_two_sided_sens = float(model_k5.pvalues[1])

print()
print(f'α̂₁^USD (k=5)      = {alpha_1_usd_sens:+.6f}')
print(f'HAC SE            = {se_alpha_1_sens:.6f}')
print(f't-stat            = {t_stat_sens:+.4f}')
print(f'p_two_sided       = {p_two_sided_sens:.6f}')

T_lag (k=5) = 23
HAC L  = floor(T_lag^(1/3)) = floor(2.8439) = 2

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.091
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.357
Date:                Sun, 17 May 2026   Prob (F-statistic):              0.280
Time:                        23:45:16   Log-Likelihood:                -35.781
No. Observations:                  23   AIC:                             77.56
Df Residuals:                      20   BIC:                             80.97
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

### Interpretation — Trio 3 (k=5 sensitivity)

Estimate and two-sided p reported above. Convergence with k=1 reinforces the verdict; divergence flags lag-dependent behavioral response (worth noting in disposition memo but does not change the primary verdict).

## Trio 4 — Power measurement on residual SD (CORRECTIONS-J + CORRECTIONS-U)

### Decision-citation block (4-part)

- **Reference:** spec §2.2.B + CORRECTIONS-J + CORRECTIONS-U.
- **Why:** post-control residual variance is the correct yardstick for MDES; using raw-LHS SD double-counts variance the token-vol control already explains. Power threshold pinned at 0.50 (lowered from 0.80 because the test is verdict-symmetric per CORRECTIONS-S — failing-to-reject is just as informative as rejecting).
- **Relevance:** re-confirms Task 11's power measurement on *this specific* regression's residuals (not the EDA-stage proxy). Pinned MC draws (B=2000) and seed (20260517) at constants level prevents post-hoc draw-tuning.
- **Connection:** if measured power < 0.50, verdict cell routes to POWER-HALT regardless of p-value (cannot trust a non-rejection on an under-powered test).

In [5]:
# Step 1: partial out |Δln Tokens| from the LHS (CORRECTIONS-J residual-SD recipe)
X_partial = sm.add_constant(x2_primary)                       # constant + token vol only
model_partial = sm.OLS(y_primary, X_partial).fit()
residuals = model_partial.resid
residual_sd = float(residuals.std(ddof=1))
print(f'Residual SD (after partialling out |Δln Tokens|) = {residual_sd:.6f}')

# Step 2: pinned MDES at 0.40 × residual SD
mdes_effect = MDES_RESID_SD * residual_sd
print(f'MDES effect size  = {MDES_RESID_SD} × residual_SD = {mdes_effect:.6f}')

# Step 3: Monte Carlo — simulate alternative-data DGPs with α₁^USD = MDES,
# refit the full HAC OLS, count two-sided rejections at α=0.05.
rng = np.random.default_rng(POWER_SEED)
B = POWER_MC_DRAWS
rejected = 0

# Fix RHS draws to actual data; simulate noise + injected MDES signal on FX vol
x1_obs = x1_primary.copy()
x2_obs = x2_primary.copy()
intercept_obs = float(model_k1.params[0])
alpha2_obs = float(model_k1.params[2])

for b in range(B):
    eps = rng.normal(loc=0.0, scale=residual_sd, size=T_lag_primary)
    y_sim = intercept_obs + mdes_effect * x1_obs + alpha2_obs * x2_obs + eps
    X_sim = sm.add_constant(np.column_stack([x1_obs, x2_obs]))
    try:
        m_sim = sm.OLS(y_sim, X_sim).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_primary})
        if m_sim.pvalues[1] < ALPHA_LEVEL:
            rejected += 1
    except (np.linalg.LinAlgError, ValueError):
        continue

measured_power = rejected / B
print()
print(f'Monte Carlo draws B    = {B}')
print(f'Rejections at α=0.05   = {rejected}')
print(f'Measured power         = {measured_power:.4f}')
print(f'Power threshold        = {POWER_THRESHOLD}')
print(f'Power gate             = {"PASS" if measured_power >= POWER_THRESHOLD else "FAIL (route POWER-HALT)"}')

Residual SD (after partialling out |Δln Tokens|) = 1.155794
MDES effect size  = 0.4 × residual_SD = 0.462317



Monte Carlo draws B    = 2000
Rejections at α=0.05   = 349
Measured power         = 0.1745
Power threshold        = 0.5
Power gate             = FAIL (route POWER-HALT)


### Interpretation — Trio 4 (power)

Measured power for the two-sided HAC test at MDES = 0.40 × residual-SD, with $B = 2000$ Monte Carlo draws under the alternative DGP, is reported above. The 0.50 floor is the pinned threshold (CORRECTIONS-J + CORRECTIONS-U); below this the test cannot reliably distinguish a true effect from noise and a non-rejection cannot be interpreted as evidence for the null.

## Verdict cell

In [6]:
# Gate evaluation
gate_p = p_two_sided_primary < ALPHA_LEVEL
gate_n = T >= N_MIN
gate_power = measured_power >= POWER_THRESHOLD

# Verdict routing
if not gate_power:
    verdict = 'POWER-HALT'
elif gate_n and gate_p:
    verdict = 'REJECT_NULL'
elif gate_n and not gate_p:
    verdict = 'FAIL_TO_REJECT'
elif (not gate_n) and gate_p:
    verdict = 'PARTIAL-REJECT'
elif (not gate_n) and not gate_p:
    verdict = 'PARTIAL-FAIL_TO_REJECT'
else:
    verdict = 'HALT'

assert verdict in VERDICT_LABELS, f'verdict {verdict!r} not in pinned label set'

print('=' * 64)
print('R4-S3-USD BEHAVIORAL TEST VERDICT')
print('=' * 64)
print(f'k=1 primary:    α̂₁^USD = {alpha_1_usd_primary:+.6f}, p_2s = {p_two_sided_primary:.6f}')
print(f'k=5 sensit.:    α̂₁^USD = {alpha_1_usd_sens:+.6f}, p_2s = {p_two_sided_sens:.6f}')
print(f'N observed:     {T}     N_MIN spec floor: {N_MIN}')
print(f'Power (resid):  {measured_power:.4f}  Threshold: {POWER_THRESHOLD}')
print('=' * 64)
print()
print('Gate evaluation:')
print(f'  p < 0.05?       {gate_p}')
print(f'  N >= 38?        {gate_n}')
print(f'  Power >= 0.50?  {gate_power}')
print()
print(f'VERDICT: {verdict}')
print()
print('Interpretation per CORRECTIONS-S:')
print('  - REJECT (full or partial) = behavioral channel EXISTS despite subscription')
print('  - FAIL_TO_REJECT (full or partial) = subscription inelasticity CONFIRMED')
print('  - Either outcome informs M-design behavioral assumptions; neither is "FAIL"')
print('=' * 64)

if verdict.startswith('PARTIAL-'):
    print()
    print('PARTIAL-* label rationale: N=29 < spec floor 38. Verdict is reported')
    print('for the record with N-below-floor explicitly disclosed. NOT a silent override.')
    print('Re-run when panel reaches N>=38 to upgrade label to REJECT_NULL/FAIL_TO_REJECT.')

R4-S3-USD BEHAVIORAL TEST VERDICT
k=1 primary:    α̂₁^USD = -18.116337, p_2s = 0.652410
k=5 sensit.:    α̂₁^USD = -35.750229, p_2s = 0.240405
N observed:     28     N_MIN spec floor: 38
Power (resid):  0.1745  Threshold: 0.5

Gate evaluation:
  p < 0.05?       False
  N >= 38?        False
  Power >= 0.50?  False

VERDICT: POWER-HALT

Interpretation per CORRECTIONS-S:
  - REJECT (full or partial) = behavioral channel EXISTS despite subscription
  - FAIL_TO_REJECT (full or partial) = subscription inelasticity CONFIRMED
  - Either outcome informs M-design behavioral assumptions; neither is "FAIL"


## Final disclaimers

**CORRECTIONS-S framing recap.** This notebook (Task 14, R4-S3-USD) is the *framework-relevant arm* of the R4-S3 family. R4-S3-COP (Task 13) was a *consistency check* — it confirmed mechanical FX pass-through but was not verdict-bearing under the (Y, M, X) framework; its CONSISTENCY-FAIL was reclassified to REGIME-CONDITIONAL per v0.2.8 §0.8 Z2 (both preconditions verified).

**Convergent evidence with R5 (descriptive) and Z-arms (multi-period + backcast).** R5 reports FX share of cost variance ≈ 0.003% under the subscription regime; both Z-arms corroborate the small-FX finding across alternative time-windows. The present notebook supplies the single inferential gate on the behavioral question ("does FX vol *affect user behavior* even though it cannot mechanically affect their USD bill?"). All four lines of evidence are independent.

**π̂ disclaimer (CORRECTIONS-Y-6).** `ephemeral_pi_share = 0.398977` is a calibrated constant, not a time-series measurement. Sensitivity to this constant is bounded by Task 11/EDA tornado plots.

**Y-9 closure.** Cost data is at 1.000000× ccusage parity (UTC-aligned). LHS and RHS are apples-to-apples on the same UTC weekday cadence.

**Anti-fishing reminder.** Pre-pinned: LAG_PRIMARY=1, LAG_SENSITIVITY=5, HAC_LAGS=⌊T^(1/3)⌋, ALPHA_LEVEL=0.05 TWO-SIDED, MDES=0.40 residual-SD, power threshold 0.50, B=2000 MC draws. None of these were tuned after seeing the result. If N<38, the verdict is honestly disclosed as PARTIAL-*, not silently labeled as if the floor were met.